# 32｜不用预制 Transformer：手写 BERT Encoder、MLM 与句对分类

本笔记实现 BERT 的 token/position/segment 三类嵌入、手写双向多头注意力编码器、与词嵌入权重绑定的 MLM head，以及 `[CLS]` 句对分类 head。我们把 **80/10/10 masking、只在被选位置计算 MLM loss、padding 不变性和双向非因果性** 都写成可运行 oracle。

> 这里的合成语料只用于验证实现可训练；不能把小样本记忆结果解释为预训练质量。

## 1. BERT 张量合同

- `input_ids/token_type_ids/attention_mask: [B,T]`；序列形如 `[CLS] A [SEP] B [SEP] [PAD]...`。
- `token_type_id=0/1` 区分句子 A/B；position id 为 `0..T-1`。
- Encoder 隐状态 `[B,T,D]`，注意力 `[B,H,T,T]`；只屏蔽 padding key，**不加 causal mask**。
- MLM label 未被选的位置为 `-100`；句对分类 logits 为 `[B,2]`。
- 不调用 `nn.MultiheadAttention`、`nn.Transformer`、`transformers` 或联网数据。

In [ ]:
import warnings  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。

import copy  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import io  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import random  # 导入本单元所需的依赖。

import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 20260811  # 计算并保存当前步骤的中间状态。
random.seed(SEED)  # 执行当前语句以推进本节示例。
torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
PAD, CLS, SEP, MASK, UNK = 0, 1, 2, 3, 4  # 计算并保存当前步骤的中间状态。
SPECIAL_IDS = {PAD, CLS, SEP, MASK, UNK}  # 计算并保存当前步骤的中间状态。
VOCAB_SIZE = 32  # 计算并保存当前步骤的中间状态。
BERT_TOKENS = ["<pad>", "<cls>", "<sep>", "<mask>", "<unk>"] + [  # 计算并保存当前步骤的中间状态。
    f"token_{index}" for index in range(5, VOCAB_SIZE)  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。

assert DEVICE.type == "cpu"  # 用受控断言验证关键不变量。
assert torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert len(SPECIAL_IDS) == 5 and max(SPECIAL_IDS) < VOCAB_SIZE  # 用受控断言验证关键不变量。
assert len(BERT_TOKENS) == VOCAB_SIZE and len(set(BERT_TOKENS)) == VOCAB_SIZE  # 用受控断言验证关键不变量。
print({"torch": torch.__version__, "seed": SEED, "vocab_size": VOCAB_SIZE})  # 执行当前语句以推进本节示例。

## 2. 逐行可复现的 80/10/10 Masking

BERT 从非 special、非 padding 的候选位置中选择监督。这里把 `[UNK]` 也视为 special：它既不能成为 MLM target，也不能被 random 10% 注入。每一行独立抽样；只要该行有普通候选且 `mlm_probability>0`，至少选择一处，避免一个样本完全没有 MLM 梯度。

对每行已选位置，用 `round` 分配约 80% `[MASK]`、10% 随机普通 token、其余保持原词，三类都预测原 token。局部 `torch.Generator` 同时驱动逐行 permutation 和随机替换；同 seed 得到相同结果且不污染全局 RNG。小行受整数取整影响，不应声称每行比例精确等于 80/10/10，统计中会保留每行计数。

In [ ]:
def mask_80_10_10(input_ids, mlm_probability, vocab_size, seed):  # 定义本节可复用的核心函数。
    if input_ids.dtype != torch.long or input_ids.ndim != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("input_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
    if input_ids.numel() and (int(input_ids.min()) < 0 or int(input_ids.max()) >= vocab_size):  # 按当前条件选择后续控制路径。
        raise ValueError("input_ids 含词表外 id")  # 遇到非法合同立即显式失败。
    if not 0.0 <= mlm_probability <= 1.0:  # 按当前条件选择后续控制路径。
        raise ValueError("mlm_probability 必须在 [0,1]")  # 遇到非法合同立即显式失败。
    ordinary_start = max(SPECIAL_IDS) + 1  # 计算并保存当前步骤的中间状态。
    if vocab_size <= ordinary_start:  # 按当前条件选择后续控制路径。
        raise ValueError("词表没有普通 token")  # 遇到非法合同立即显式失败。
    eligible = torch.ones_like(input_ids, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    for special_id in SPECIAL_IDS:  # 遍历输入元素以累积或检查结果。
        eligible &= input_ids.ne(special_id)  # 计算并保存当前步骤的中间状态。
    corrupted = input_ids.clone()  # 计算并保存当前步骤的中间状态。
    labels = torch.full_like(input_ids, -100)  # 计算并保存当前步骤的中间状态。
    selected = torch.zeros_like(input_ids, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    rng = torch.Generator(device=input_ids.device).manual_seed(int(seed))  # 计算并保存当前步骤的中间状态。
    totals = {"selected": 0, "mask": 0, "random": 0, "same": 0}  # 计算并保存当前步骤的中间状态。
    per_row = []  # 计算并保存当前步骤的中间状态。

    for row in range(input_ids.shape[0]):  # 遍历输入元素以累积或检查结果。
        candidate_idx = eligible[row].nonzero(as_tuple=False).squeeze(1)  # 计算并保存当前步骤的中间状态。
        n_select = int(round(candidate_idx.numel() * mlm_probability))  # 计算并保存当前步骤的中间状态。
        if mlm_probability > 0 and candidate_idx.numel() > 0:  # 按当前条件选择后续控制路径。
            n_select = max(1, n_select)  # 计算并保存当前步骤的中间状态。
        order = candidate_idx[torch.randperm(candidate_idx.numel(), generator=rng,  # 计算并保存当前步骤的中间状态。
                                             device=input_ids.device)]  # 计算并保存当前步骤的中间状态。
        chosen = order[:n_select]  # 计算并保存当前步骤的中间状态。
        n_mask = int(round(0.8 * n_select))  # 计算并保存当前步骤的中间状态。
        n_random = int(round(0.1 * n_select))  # 计算并保存当前步骤的中间状态。
        n_same = n_select - n_mask - n_random  # 计算并保存当前步骤的中间状态。
        labels[row, chosen] = input_ids[row, chosen]  # 计算并保存当前步骤的中间状态。
        selected[row, chosen] = True  # 计算并保存当前步骤的中间状态。
        corrupted[row, chosen[:n_mask]] = MASK  # 计算并保存当前步骤的中间状态。
        random_idx = chosen[n_mask:n_mask + n_random]  # 计算并保存当前步骤的中间状态。
        if random_idx.numel():  # 按当前条件选择后续控制路径。
            corrupted[row, random_idx] = torch.randint(  # 计算并保存当前步骤的中间状态。
                ordinary_start, vocab_size, (random_idx.numel(),), generator=rng,  # 计算并保存当前步骤的中间状态。
                device=input_ids.device,  # 计算并保存当前步骤的中间状态。
            )  # 执行当前语句以推进本节示例。
        row_stats = {"selected": n_select, "mask": n_mask,  # 计算并保存当前步骤的中间状态。
                     "random": n_random, "same": n_same}  # 执行当前语句以推进本节示例。
        per_row.append(row_stats)  # 执行当前语句以推进本节示例。
        for key in totals:  # 遍历输入元素以累积或检查结果。
            totals[key] += row_stats[key]  # 计算并保存当前步骤的中间状态。
    stats = {**totals, "per_row": per_row}  # 计算并保存当前步骤的中间状态。
    return corrupted, labels, selected, stats  # 返回当前分支计算出的结果。


mask_probe = torch.arange(5, 25).repeat(10, 1)  # 计算并保存当前步骤的中间状态。
mask_probe[:, 0] = CLS  # 计算并保存当前步骤的中间状态。
mask_probe[:, -1] = SEP  # 计算并保存当前步骤的中间状态。
c1, y1, s1, stats1 = mask_80_10_10(mask_probe, 0.5, VOCAB_SIZE, seed=19)  # 计算并保存当前步骤的中间状态。
c2, y2, s2, stats2 = mask_80_10_10(mask_probe, 0.5, VOCAB_SIZE, seed=19)  # 计算并保存当前步骤的中间状态。
assert torch.equal(c1, c2) and torch.equal(y1, y2) and torch.equal(s1, s2)  # 用受控断言验证关键不变量。
assert not bool(s1[:, [0, -1]].any()) and bool(s1.any(1).all())  # 用受控断言验证关键不变量。
assert stats1["mask"] + stats1["random"] + stats1["same"] == stats1["selected"]  # 用受控断言验证关键不变量。
assert stats1 == stats2 and stats1["selected"] > 0  # 用受控断言验证关键不变量。

# 每行至少一处、UNK 排除、random 只落在普通 token 范围、空候选保持空。
row_probe = torch.tensor([[CLS, 5, SEP], [CLS, 6, SEP]])  # 计算并保存当前步骤的中间状态。
row_corrupt, row_labels, row_selected, row_stats = mask_80_10_10(  # 计算并保存当前步骤的中间状态。
    row_probe, 0.5, VOCAB_SIZE, seed=9  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert row_selected.sum(1).tolist() == [1, 1]  # 用受控断言验证关键不变量。
special_probe = torch.tensor([[PAD, CLS, SEP, MASK, UNK]])  # 计算并保存当前步骤的中间状态。
special_corrupt, special_labels, special_selected, special_stats = mask_80_10_10(  # 计算并保存当前步骤的中间状态。
    special_probe, 1.0, VOCAB_SIZE, seed=3  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert not bool(special_selected.any()) and special_stats["selected"] == 0  # 用受控断言验证关键不变量。
assert torch.equal(special_corrupt, special_probe) and bool(special_labels.eq(-100).all())  # 用受控断言验证关键不变量。
ordinary_probe = torch.tensor([[CLS] + list(range(5, 15)) + [SEP]])  # 计算并保存当前步骤的中间状态。
ordinary_corrupt, _, _, exact_stats = mask_80_10_10(ordinary_probe, 1.0, VOCAB_SIZE, seed=46)  # 计算并保存当前步骤的中间状态。
assert exact_stats["selected"] == 10 and exact_stats["mask"] == 8  # 用受控断言验证关键不变量。
assert exact_stats["random"] == 1 and exact_stats["same"] == 1  # 用受控断言验证关键不变量。
assert not bool(ordinary_corrupt.eq(UNK).any())  # 用受控断言验证关键不变量。

## 3. 双向多头注意力

$$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M_{pad}\right)V.$$

与 GPT 的关键区别是没有下三角矩阵：任意有效位置都可以读取其左右上下文。padding key 被屏蔽，padding query 的输出再归零。注意力矩阵占 $O(BHT^2)$ 空间，这是 BERT 长序列的主要瓶颈。

In [ ]:
def prefix_mask_or_raise(mask):  # 定义本节可复用的核心函数。
    if mask.ndim != 2 or mask.dtype != torch.bool:  # 按当前条件选择后续控制路径。
        raise ValueError("attention_mask 必须为二维 bool")  # 遇到非法合同立即显式失败。
    if bool((~mask.any(1)).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("每条序列至少有一个有效 token")  # 遇到非法合同立即显式失败。
    if mask.shape[1] > 1 and bool((mask[:, 1:] & ~mask[:, :-1]).any()):  # 按当前条件选择后续控制路径。
        raise ValueError("有效 token 必须左对齐")  # 遇到非法合同立即显式失败。


class ManualBidirectionalAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, n_heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if d_model <= 0 or n_heads <= 0 or d_model % n_heads:  # 按当前条件选择后续控制路径。
            raise ValueError("d_model 必须能被 n_heads 整除")  # 遇到非法合同立即显式失败。
        self.d_model, self.n_heads, self.head_dim = d_model, n_heads, d_model // n_heads  # 计算并保存当前步骤的中间状态。
        self.q_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.k_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.v_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。

    def _split(self, x):  # 定义本节可复用的核心函数。
        B, T, _ = x.shape  # 计算并保存当前步骤的中间状态。
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # 返回当前分支计算出的结果。

    def forward(self, x, attention_mask):  # 定义本节可复用的核心函数。
        B, T, D = x.shape  # 计算并保存当前步骤的中间状态。
        if D != self.d_model or attention_mask.shape != (B, T):  # 按当前条件选择后续控制路径。
            raise ValueError("attention 输入形状错误")  # 遇到非法合同立即显式失败。
        prefix_mask_or_raise(attention_mask)  # 执行当前语句以推进本节示例。
        q, k, v = self._split(self.q_proj(x)), self._split(self.k_proj(x)), self._split(self.v_proj(x))  # 计算并保存当前步骤的中间状态。
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)  # 计算并保存当前步骤的中间状态。
        allowed = attention_mask[:, None, None, :]  # 计算并保存当前步骤的中间状态。
        scores = scores.masked_fill(~allowed, torch.finfo(scores.dtype).min)  # 计算并保存当前步骤的中间状态。
        weights = torch.softmax(scores, dim=-1)  # 计算并保存当前步骤的中间状态。
        # padding query 不应向调试/解释消费者暴露一行伪概率。
        weights = weights * attention_mask[:, None, :, None]  # 计算并保存当前步骤的中间状态。
        context = weights @ v  # 计算并保存当前步骤的中间状态。
        context = context.transpose(1, 2).contiguous().view(B, T, D)  # 计算并保存当前步骤的中间状态。
        output = self.out_proj(context) * attention_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return output, weights  # 返回当前分支计算出的结果。


# 构造一个可解释 oracle：q/k 全零 => 对所有有效 key 均匀加权；修改右侧 token 必须影响左侧输出。
oracle_attn = ManualBidirectionalAttention(4, 2)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    oracle_attn.q_proj.weight.zero_(); oracle_attn.q_proj.bias.zero_()  # 执行当前语句以推进本节示例。
    oracle_attn.k_proj.weight.zero_(); oracle_attn.k_proj.bias.zero_()  # 执行当前语句以推进本节示例。
    oracle_attn.v_proj.weight.copy_(torch.eye(4)); oracle_attn.v_proj.bias.zero_()  # 执行当前语句以推进本节示例。
    oracle_attn.out_proj.weight.copy_(torch.eye(4)); oracle_attn.out_proj.bias.zero_()  # 执行当前语句以推进本节示例。
oracle_x = torch.tensor([[[1., 0., 0., 0.], [0., 1., 0., 0.], [0., 0., 1., 0.]]])  # 计算并保存当前步骤的中间状态。
oracle_mask = torch.ones(1, 3, dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
oracle_changed = oracle_x.clone(); oracle_changed[:, 2, 2] = 5.0  # 计算并保存当前步骤的中间状态。
left_before = oracle_attn(oracle_x, oracle_mask)[0][:, 0]  # 计算并保存当前步骤的中间状态。
left_after = oracle_attn(oracle_changed, oracle_mask)[0][:, 0]  # 计算并保存当前步骤的中间状态。
assert not torch.allclose(left_before, left_after)  # 用受控断言验证关键不变量。
assert torch.allclose(oracle_attn(oracle_x, oracle_mask)[1].sum(-1), torch.ones(1, 2, 3))  # 用受控断言验证关键不变量。

## 4. Embeddings、Post-Norm Encoder 与双任务 head

BERT 输入为三类嵌入之和：

$$h_i^0=E_{token}(x_i)+E_{position}(i)+E_{segment}(s_i).$$

原始 BERT block 使用 Post-Norm：`LN(x + Attention(x))`，再做 `LN(x + FFN(x))`。MLM head 先经过 dense、GELU、LayerNorm，再投影到词表；输出矩阵与 token embedding 共享。句对分类读取最终 `[CLS]` 表示。

In [ ]:
class BertEmbeddings(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, max_len, d_model, type_vocab_size=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.max_len, self.type_vocab_size = max_len, type_vocab_size  # 计算并保存当前步骤的中间状态。
        self.token = nn.Embedding(vocab_size, d_model, padding_idx=PAD)  # 计算并保存当前步骤的中间状态。
        self.position = nn.Embedding(max_len, d_model)  # 计算并保存当前步骤的中间状态。
        self.segment = nn.Embedding(type_vocab_size, d_model)  # 计算并保存当前步骤的中间状态。
        self.norm = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。

    def forward(self, input_ids, token_type_ids, attention_mask):  # 定义本节可复用的核心函数。
        B, T = input_ids.shape  # 计算并保存当前步骤的中间状态。
        if token_type_ids.shape != (B, T) or attention_mask.shape != (B, T):  # 按当前条件选择后续控制路径。
            raise ValueError("embedding 输入形状不一致")  # 遇到非法合同立即显式失败。
        if T > self.max_len or input_ids.dtype != torch.long or token_type_ids.dtype != torch.long:  # 按当前条件选择后续控制路径。
            raise ValueError("长度或 dtype 错误")  # 遇到非法合同立即显式失败。
        if token_type_ids.numel() and (int(token_type_ids.min()) < 0 or int(token_type_ids.max()) >= self.type_vocab_size):  # 按当前条件选择后续控制路径。
            raise ValueError("segment id 越界")  # 遇到非法合同立即显式失败。
        pos = torch.arange(T, device=input_ids.device)[None]  # 计算并保存当前步骤的中间状态。
        x = self.token(input_ids) + self.position(pos) + self.segment(token_type_ids)  # 计算并保存当前步骤的中间状态。
        return self.norm(x) * attention_mask.unsqueeze(-1)  # 返回当前分支计算出的结果。


class BertEncoderBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, n_heads, ffn_dim):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.attention = ManualBidirectionalAttention(d_model, n_heads)  # 计算并保存当前步骤的中间状态。
        self.norm1 = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.ffn = nn.Sequential(nn.Linear(d_model, ffn_dim), nn.GELU(), nn.Linear(ffn_dim, d_model))  # 计算并保存当前步骤的中间状态。
        self.norm2 = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, attention_mask):  # 定义本节可复用的核心函数。
        attended, weights = self.attention(x, attention_mask)  # 计算并保存当前步骤的中间状态。
        x = self.norm1(x + attended) * attention_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        x = self.norm2(x + self.ffn(x)) * attention_mask.unsqueeze(-1)  # 计算并保存当前步骤的中间状态。
        return x, weights  # 返回当前分支计算出的结果。


class ScratchBert(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size, max_len, d_model=32, n_heads=4, n_layers=2, ffn_dim=64):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        if vocab_size <= max(SPECIAL_IDS) or max_len <= 2 or n_layers <= 0:  # 按当前条件选择后续控制路径。
            raise ValueError("BERT 配置错误")  # 遇到非法合同立即显式失败。
        self.config = dict(vocab_size=vocab_size, max_len=max_len, d_model=d_model,  # 计算并保存当前步骤的中间状态。
                           n_heads=n_heads, n_layers=n_layers, ffn_dim=ffn_dim)  # 计算并保存当前步骤的中间状态。
        self.embeddings = BertEmbeddings(vocab_size, max_len, d_model)  # 计算并保存当前步骤的中间状态。
        self.layers = nn.ModuleList([BertEncoderBlock(d_model, n_heads, ffn_dim) for _ in range(n_layers)])  # 计算并保存当前步骤的中间状态。
        self.mlm_dense = nn.Linear(d_model, d_model)  # 计算并保存当前步骤的中间状态。
        self.mlm_norm = nn.LayerNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.mlm_decoder = nn.Linear(d_model, vocab_size, bias=True)  # 计算并保存当前步骤的中间状态。
        self.mlm_decoder.weight = self.embeddings.token.weight  # 计算并保存当前步骤的中间状态。
        self.sentence_classifier = nn.Linear(d_model, 2)  # 计算并保存当前步骤的中间状态。

    def forward(self, input_ids, token_type_ids, attention_mask):  # 定义本节可复用的核心函数。
        if input_ids.ndim != 2 or input_ids.dtype != torch.long:  # 按当前条件选择后续控制路径。
            raise ValueError("input_ids 必须是二维 long")  # 遇到非法合同立即显式失败。
        if input_ids.numel() and (int(input_ids.min()) < 0 or int(input_ids.max()) >= self.config["vocab_size"]):  # 按当前条件选择后续控制路径。
            raise ValueError("token id 越界")  # 遇到非法合同立即显式失败。
        prefix_mask_or_raise(attention_mask)  # 执行当前语句以推进本节示例。
        x = self.embeddings(input_ids, token_type_ids, attention_mask)  # 计算并保存当前步骤的中间状态。
        weights = []  # 计算并保存当前步骤的中间状态。
        for layer in self.layers:  # 遍历输入元素以累积或检查结果。
            x, w = layer(x, attention_mask)  # 计算并保存当前步骤的中间状态。
            weights.append(w)  # 执行当前语句以推进本节示例。
        mlm_hidden = self.mlm_norm(F.gelu(self.mlm_dense(x)))  # 计算并保存当前步骤的中间状态。
        mlm_logits = self.mlm_decoder(mlm_hidden)  # 计算并保存当前步骤的中间状态。
        sentence_logits = self.sentence_classifier(x[:, 0])  # 计算并保存当前步骤的中间状态。
        return {"hidden": x, "mlm_logits": mlm_logits,  # 返回当前分支计算出的结果。
                "sentence_logits": sentence_logits, "attentions": weights}  # 执行当前语句以推进本节示例。

In [ ]:
bert = ScratchBert(VOCAB_SIZE, max_len=12, d_model=24, n_heads=4, n_layers=2, ffn_dim=48)  # 计算并保存当前步骤的中间状态。
ids0 = torch.tensor([[CLS, 5, 6, SEP, 7, 8, SEP, PAD], [CLS, 9, SEP, 10, 11, 12, SEP, PAD]])  # 计算并保存当前步骤的中间状态。
types0 = torch.tensor([[0, 0, 0, 0, 1, 1, 1, 0], [0, 0, 0, 1, 1, 1, 1, 0]])  # 计算并保存当前步骤的中间状态。
mask0 = ids0.ne(PAD)  # 计算并保存当前步骤的中间状态。
out0 = bert(ids0, types0, mask0)  # 计算并保存当前步骤的中间状态。
assert out0["hidden"].shape == (2, 8, 24)  # 用受控断言验证关键不变量。
assert out0["mlm_logits"].shape == (2, 8, VOCAB_SIZE)  # 用受控断言验证关键不变量。
assert out0["sentence_logits"].shape == (2, 2)  # 用受控断言验证关键不变量。
assert len(out0["attentions"]) == 2  # 用受控断言验证关键不变量。
assert bert.mlm_decoder.weight is bert.embeddings.token.weight  # 用受控断言验证关键不变量。
assert torch.count_nonzero(out0["hidden"][:, -1]) == 0  # 用受控断言验证关键不变量。

## 5. Padding 不变性与联合损失

同一有效前缀后面的 PAD id 即便被替换成任意普通 id，只要 `attention_mask=False`，有效位置输出也不得变化。MLM loss 只选 `labels != -100` 的位置；不能把未选 token 或 padding 也计入分母。

联合目标写作 $\mathcal L=\mathcal L_{MLM}+\lambda\mathcal L_{pair}$。真实训练应分别记录两项、mask 数、句对类别分布和梯度尺度，避免某个任务静默主导。

In [ ]:
bert.eval()  # 执行当前语句以推进本节示例。
pad_changed = ids0.clone()  # 计算并保存当前步骤的中间状态。
pad_changed[:, -1] = torch.tensor([17, 19])  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    base_out = bert(ids0, types0, mask0)  # 计算并保存当前步骤的中间状态。
    changed_out = bert(pad_changed, types0, mask0)  # 计算并保存当前步骤的中间状态。
assert torch.equal(base_out["hidden"][mask0], changed_out["hidden"][mask0])  # 用受控断言验证关键不变量。
assert torch.equal(base_out["sentence_logits"], changed_out["sentence_logits"])  # 用受控断言验证关键不变量。
assert torch.count_nonzero(base_out["attentions"][0][:, :, -1]) == 0  # 用受控断言验证关键不变量。


def pretraining_loss(outputs, mlm_labels, sentence_labels, sentence_weight=1.0):  # 定义本节可复用的核心函数。
    selected = mlm_labels.ne(-100)  # 计算并保存当前步骤的中间状态。
    if not bool(selected.any()):  # 按当前条件选择后续控制路径。
        raise ValueError("当前 batch 没有 MLM 监督位置")  # 遇到非法合同立即显式失败。
    if sentence_labels.shape != (mlm_labels.shape[0],):  # 按当前条件选择后续控制路径。
        raise ValueError("句对标签形状错误")  # 遇到非法合同立即显式失败。
    mlm_loss = F.cross_entropy(outputs["mlm_logits"][selected], mlm_labels[selected])  # 计算并保存当前步骤的中间状态。
    sentence_loss = F.cross_entropy(outputs["sentence_logits"], sentence_labels)  # 计算并保存当前步骤的中间状态。
    return mlm_loss + sentence_weight * sentence_loss, mlm_loss, sentence_loss  # 返回当前分支计算出的结果。

corrupt0, labels0, selected0, _ = mask_80_10_10(ids0, 0.4, VOCAB_SIZE, seed=31)  # 计算并保存当前步骤的中间状态。
loss_out = bert(corrupt0, types0, mask0)  # 计算并保存当前步骤的中间状态。
total0, mlm0, sent0 = pretraining_loss(loss_out, labels0, torch.tensor([1, 0]), 0.7)  # 计算并保存当前步骤的中间状态。
manual_mlm = F.cross_entropy(loss_out["mlm_logits"][selected0], ids0[selected0])  # 计算并保存当前步骤的中间状态。
assert torch.allclose(mlm0, manual_mlm)  # 用受控断言验证关键不变量。
assert torch.allclose(total0, mlm0 + 0.7 * sent0)  # 用受控断言验证关键不变量。
assert torch.isfinite(total0)  # 用受控断言验证关键不变量。

## 6. 合成句对与受控训练

标签 1 的句子 B 延续 A 的主题 token，标签 0 则换到另一个主题。这个人为规则只用于确认 MLM head、`[CLS]` head、共享 embedding 和反向传播能协同工作。mask 在训练前固定生成，使前后 loss 可直接比较；真实 BERT 预训练通常每个 epoch 动态重采样 mask。

In [ ]:
def make_pairs():  # 定义本节可复用的核心函数。
    rows, types, labels = [], [], []  # 计算并保存当前步骤的中间状态。
    for i in range(16):  # 遍历输入元素以累积或检查结果。
        topic = 5 + (i % 4)  # 计算并保存当前步骤的中间状态。
        positive = i % 2 == 0  # 计算并保存当前步骤的中间状态。
        other = topic if positive else 5 + ((i + 1) % 4)  # 计算并保存当前步骤的中间状态。
        a = [topic, 12 + (i % 5), 17 + (i % 4)]  # 计算并保存当前步骤的中间状态。
        b = [other, 21 + (i % 5)]  # 计算并保存当前步骤的中间状态。
        row = [CLS] + a + [SEP] + b + [SEP]  # 计算并保存当前步骤的中间状态。
        typ = [0] * (len(a) + 2) + [1] * (len(b) + 1)  # 计算并保存当前步骤的中间状态。
        rows.append(row + [PAD] * (10 - len(row)))  # 执行当前语句以推进本节示例。
        types.append(typ + [0] * (10 - len(typ)))  # 执行当前语句以推进本节示例。
        labels.append(int(positive))  # 执行当前语句以推进本节示例。
    ids = torch.tensor(rows, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    token_types = torch.tensor(types, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    return ids, token_types, ids.ne(PAD), torch.tensor(labels, dtype=torch.long)  # 返回当前分支计算出的结果。

pair_ids, pair_types, pair_mask, pair_labels = make_pairs()  # 计算并保存当前步骤的中间状态。
corrupted_ids, mlm_labels, selected_mask, masking_stats = mask_80_10_10(  # 计算并保存当前步骤的中间状态。
    pair_ids, mlm_probability=0.35, vocab_size=VOCAB_SIZE, seed=SEED + 9  # 计算并保存当前步骤的中间状态。
)  # 执行当前语句以推进本节示例。
assert not bool(selected_mask[pair_ids.eq(PAD) | pair_ids.eq(CLS) | pair_ids.eq(SEP)].any())  # 用受控断言验证关键不变量。
assert int(mlm_labels.ne(-100).sum()) == masking_stats["selected"]  # 用受控断言验证关键不变量。
assert set(pair_labels.tolist()) == {0, 1}  # 用受控断言验证关键不变量。

In [ ]:
torch.manual_seed(SEED + 1)  # 执行当前语句以推进本节示例。
bert = ScratchBert(VOCAB_SIZE, max_len=12, d_model=24, n_heads=4, n_layers=2, ffn_dim=48)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.AdamW(bert.parameters(), lr=0.02, weight_decay=0.0)  # 计算并保存当前步骤的中间状态。

bert.train()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    initial_total = float(pretraining_loss(  # 计算并保存当前步骤的中间状态。
        bert(corrupted_ids, pair_types, pair_mask), mlm_labels, pair_labels, 0.7  # 执行当前语句以推进本节示例。
    )[0])  # 执行当前语句以推进本节示例。
for step in range(80):  # 遍历输入元素以累积或检查结果。
    optimizer.zero_grad(set_to_none=True)  # 计算并保存当前步骤的中间状态。
    outputs = bert(corrupted_ids, pair_types, pair_mask)  # 计算并保存当前步骤的中间状态。
    total, mlm_part, pair_part = pretraining_loss(outputs, mlm_labels, pair_labels, 0.7)  # 计算并保存当前步骤的中间状态。
    total.backward()  # 执行当前语句以推进本节示例。
    grad_norm = torch.nn.utils.clip_grad_norm_(bert.parameters(), 1.0)  # 计算并保存当前步骤的中间状态。
    assert torch.isfinite(grad_norm)  # 用受控断言验证关键不变量。
    optimizer.step()  # 执行当前语句以推进本节示例。

bert.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    trained_out = bert(corrupted_ids, pair_types, pair_mask)  # 计算并保存当前步骤的中间状态。
    final_total, final_mlm, final_pair = pretraining_loss(trained_out, mlm_labels, pair_labels, 0.7)  # 计算并保存当前步骤的中间状态。
    mlm_accuracy = (trained_out["mlm_logits"][selected_mask].argmax(-1) == mlm_labels[selected_mask]).float().mean()  # 计算并保存当前步骤的中间状态。
    pair_accuracy = (trained_out["sentence_logits"].argmax(-1) == pair_labels).float().mean()  # 计算并保存当前步骤的中间状态。
assert float(final_total) < initial_total * 0.25  # 用受控断言验证关键不变量。
assert float(mlm_accuracy) > 0.95 and float(pair_accuracy) > 0.95  # 用受控断言验证关键不变量。
print({"loss": [round(initial_total, 4), round(float(final_total), 4)],  # 执行当前语句以推进本节示例。
       "mlm_acc": round(float(mlm_accuracy), 3), "pair_acc": round(float(pair_accuracy), 3)})  # 执行当前语句以推进本节示例。

## 7. 评估应该分层，而不是只看总 loss

- MLM：仅在被选位置报告 token accuracy / top-k accuracy，并按词频、词类、序列位置切片。
- 句对任务：报告 accuracy 之外，还需混淆矩阵、宏平均 F1、类别比例和置信度校准。
- 数据切分必须按文档、时间或实体隔离，不能让同一句话的改写跨 train/test。
- padding invariance、双向干预、special token 排除和相同 seed 复现属于单元测试，应在模型质量评估之前运行。

受控训练达到 100% 只说明模型记住了 16 个样本；对真实语料的结论必须来自独立验证集。

## 8. 完整语义制品与发布者 registry

BERT checkpoint 必须绑定完整 token 顺序、固定且互异的 special IDs、句对 label map、逐行 masking recipe、原始训练数据与 split、预处理、网络配置和权重。只在 package 内保存 SHA-256 不是信任锚：攻击者能整体替换内容并重算它。

下面由发布者侧将 `(artifact_id, version) -> immutable manifest digest` 放入只读 registry。loader 先验证 registry，再验证原始 state bytes 和按 tensor key/dtype/shape/bytes 计算的权重摘要，最后做 config/vocab/label/data 的语义交叉校验。生产 registry 应是受控数据库或签名透明日志，而不是请求参数。

In [ ]:
from types import MappingProxyType  # 导入本单元所需的依赖。

def stable_hash(obj):  # 定义本节可复用的核心函数。
    raw = json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode()  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw).hexdigest()  # 返回当前分支计算出的结果。


def bert_tensor_state_hash(state_dict):  # 定义本节可复用的核心函数。
    digest = hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for key in sorted(state_dict):  # 遍历输入元素以累积或检查结果。
        tensor = state_dict[key]  # 计算并保存当前步骤的中间状态。
        if not isinstance(tensor, torch.Tensor):  # 按当前条件选择后续控制路径。
            raise TypeError("state_dict 只能包含 tensor")  # 遇到非法合同立即显式失败。
        value = tensor.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        descriptor = {"key": key, "dtype": str(value.dtype), "shape": list(value.shape)}  # 计算并保存当前步骤的中间状态。
        digest.update(json.dumps(descriptor, sort_keys=True, separators=(",", ":")).encode())  # 计算并保存当前步骤的中间状态。
        digest.update(value.numpy().tobytes(order="C"))  # 计算并保存当前步骤的中间状态。
    return digest.hexdigest()  # 返回当前分支计算出的结果。


def validate_bert_manifest(manifest):  # 定义本节可复用的核心函数。
    required = {"schema", "artifact_id", "version", "subject", "config", "vocab",  # 计算并保存当前步骤的中间状态。
                "label_map", "preprocess", "training_snapshot", "state_bytes_sha256",  # 执行当前语句以推进本节示例。
                "state_tensor_sha256"}  # 执行当前语句以推进本节示例。
    if not isinstance(manifest, dict) or set(manifest) != required:  # 按当前条件选择后续控制路径。
        raise ValueError("BERT manifest 字段不完整")  # 遇到非法合同立即显式失败。
    if manifest["schema"] != "scratch-bert-v2" or not manifest["artifact_id"] or not manifest["version"]:  # 按当前条件选择后续控制路径。
        raise ValueError("artifact 身份字段错误")  # 遇到非法合同立即显式失败。
    config = manifest["config"]  # 计算并保存当前步骤的中间状态。
    if set(config) != {"vocab_size", "max_len", "d_model", "n_heads", "n_layers", "ffn_dim"}:  # 按当前条件选择后续控制路径。
        raise ValueError("BERT config 字段错误")  # 遇到非法合同立即显式失败。
    vocab = manifest["vocab"]  # 计算并保存当前步骤的中间状态。
    if set(vocab) != {"kind", "tokens", "special_ids"} or vocab["kind"] != "demo-token-list-v1":  # 按当前条件选择后续控制路径。
        raise ValueError("vocab schema 错误")  # 遇到非法合同立即显式失败。
    tokens = vocab["tokens"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(tokens, list) or len(tokens) != len(set(tokens)) or len(tokens) != config["vocab_size"]:  # 按当前条件选择后续控制路径。
        raise ValueError("完整 token 顺序与 vocab_size 不一致")  # 遇到非法合同立即显式失败。
    expected_special = {"pad": PAD, "cls": CLS, "sep": SEP, "mask": MASK, "unk": UNK}  # 计算并保存当前步骤的中间状态。
    if vocab["special_ids"] != expected_special:  # 按当前条件选择后续控制路径。
        raise ValueError("special id 合同错误")  # 遇到非法合同立即显式失败。
    special_values = list(vocab["special_ids"].values())  # 计算并保存当前步骤的中间状态。
    if len(special_values) != len(set(special_values)) or any(  # 按当前条件选择后续控制路径。
        not 0 <= value < len(tokens) for value in special_values  # 计算并保存当前步骤的中间状态。
    ):  # 执行当前语句以推进本节示例。
        raise ValueError("special id 必须唯一且位于词表内")  # 遇到非法合同立即显式失败。
    if tokens[:5] != ["<pad>", "<cls>", "<sep>", "<mask>", "<unk>"]:  # 按当前条件选择后续控制路径。
        raise ValueError("special token 顺序错误")  # 遇到非法合同立即显式失败。
    labels = manifest["label_map"]  # 计算并保存当前步骤的中间状态。
    if labels != ["not_next", "is_next"] or len(labels) != len(set(labels)):  # 按当前条件选择后续控制路径。
        raise ValueError("句对 label map 错误")  # 遇到非法合同立即显式失败。
    preprocess = manifest["preprocess"]  # 计算并保存当前步骤的中间状态。
    expected_preprocess = {"padding": "right", "pad_id": PAD, "max_len": config["max_len"],  # 计算并保存当前步骤的中间状态。
                           "token_type_vocab_size": 2, "attention_mask_true": "valid"}  # 执行当前语句以推进本节示例。
    if preprocess != expected_preprocess:  # 按当前条件选择后续控制路径。
        raise ValueError("BERT 预处理快照错误")  # 遇到非法合同立即显式失败。
    snapshot = manifest["training_snapshot"]  # 计算并保存当前步骤的中间状态。
    if snapshot.get("vocab_sha256") != stable_hash(vocab):  # 按当前条件选择后续控制路径。
        raise ValueError("训练快照 vocab 指纹错误")  # 遇到非法合同立即显式失败。
    dataset = snapshot.get("dataset", {})  # 计算并保存当前步骤的中间状态。
    ids, types, sentence_labels = (  # 计算并保存当前步骤的中间状态。
        dataset.get("input_ids"), dataset.get("token_type_ids"), dataset.get("sentence_labels")  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    if not isinstance(ids, list) or not ids or not (len(ids) == len(types) == len(sentence_labels)):  # 按当前条件选择后续控制路径。
        raise ValueError("训练句对快照长度错误")  # 遇到非法合同立即显式失败。
    split = dataset.get("split", {})  # 计算并保存当前步骤的中间状态。
    if set(split) != {"train", "validation", "test"}:  # 按当前条件选择后续控制路径。
        raise ValueError("训练 split 字段错误")  # 遇到非法合同立即显式失败。
    indices = split["train"] + split["validation"] + split["test"]  # 计算并保存当前步骤的中间状态。
    if len(indices) != len(set(indices)) or sorted(indices) != list(range(len(ids))):  # 按当前条件选择后续控制路径。
        raise ValueError("训练 split 必须互斥且完整")  # 遇到非法合同立即显式失败。
    for row, type_row, label in zip(ids, types, sentence_labels):  # 遍历输入元素以累积或检查结果。
        if not (len(row) == len(type_row) and 3 <= len(row) <= config["max_len"]):  # 按当前条件选择后续控制路径。
            raise ValueError("训练 row shape 错误")  # 遇到非法合同立即显式失败。
        if any(not isinstance(token, int) or token < 0 or token >= config["vocab_size"] for token in row):  # 按当前条件选择后续控制路径。
            raise ValueError("训练 token id 越界")  # 遇到非法合同立即显式失败。
        if any(value not in (0, 1) for value in type_row) or label not in (0, 1):  # 按当前条件选择后续控制路径。
            raise ValueError("训练 segment/label 越界")  # 遇到非法合同立即显式失败。
        valid = [token != PAD for token in row]  # 计算并保存当前步骤的中间状态。
        if any(valid[i] and not valid[i - 1] for i in range(1, len(valid))):  # 按当前条件选择后续控制路径。
            raise ValueError("训练 row 含 padding hole")  # 遇到非法合同立即显式失败。
    expected_masking = {"strategy": "per-row-80-10-10", "mlm_probability": 0.35,  # 计算并保存当前步骤的中间状态。
                        "seed": SEED + 9, "excluded_ids": sorted(SPECIAL_IDS)}  # 执行当前语句以推进本节示例。
    if snapshot.get("masking") != expected_masking:  # 按当前条件选择后续控制路径。
        raise ValueError("masking recipe 快照错误")  # 遇到非法合同立即显式失败。
    expected_recipe = {"optimizer": "AdamW", "steps": 80, "lr": 0.02,  # 计算并保存当前步骤的中间状态。
                       "weight_decay": 0.0, "clip_grad_norm": 1.0,  # 执行当前语句以推进本节示例。
                       "sentence_weight": 0.7, "seed": SEED + 1,  # 执行当前语句以推进本节示例。
                       "purpose": "controlled-overfit"}  # 执行当前语句以推进本节示例。
    if snapshot.get("recipe") != expected_recipe:  # 按当前条件选择后续控制路径。
        raise ValueError("训练 recipe 快照错误")  # 遇到非法合同立即显式失败。


def package_bert(model, vocab_spec, label_map, subject, artifact_id, version, training_snapshot):  # 定义本节可复用的核心函数。
    buffer = io.BytesIO()  # 计算并保存当前步骤的中间状态。
    torch.save(model.state_dict(), buffer)  # 执行当前语句以推进本节示例。
    state_bytes = buffer.getvalue()  # 计算并保存当前步骤的中间状态。
    manifest = {  # 计算并保存当前步骤的中间状态。
        "schema": "scratch-bert-v2", "artifact_id": artifact_id, "version": version,  # 执行当前语句以推进本节示例。
        "subject": subject, "config": copy.deepcopy(model.config),  # 执行当前语句以推进本节示例。
        "vocab": copy.deepcopy(vocab_spec), "label_map": list(label_map),  # 执行当前语句以推进本节示例。
        "preprocess": {"padding": "right", "pad_id": PAD, "max_len": model.config["max_len"],  # 执行当前语句以推进本节示例。
                       "token_type_vocab_size": 2, "attention_mask_true": "valid"},  # 执行当前语句以推进本节示例。
        "training_snapshot": copy.deepcopy(training_snapshot),  # 执行当前语句以推进本节示例。
        "state_bytes_sha256": hashlib.sha256(state_bytes).hexdigest(),  # 执行当前语句以推进本节示例。
        "state_tensor_sha256": bert_tensor_state_hash(model.state_dict()),  # 执行当前语句以推进本节示例。
    }  # 执行当前语句以推进本节示例。
    validate_bert_manifest(manifest)  # 执行当前语句以推进本节示例。
    return {"manifest": manifest, "manifest_sha256": stable_hash(manifest),  # 返回当前分支计算出的结果。
            "state_bytes": state_bytes}  # 执行当前语句以推进本节示例。


vocab_spec = {"kind": "demo-token-list-v1", "tokens": BERT_TOKENS,  # 计算并保存当前步骤的中间状态。
              "special_ids": {"pad": PAD, "cls": CLS, "sep": SEP,  # 执行当前语句以推进本节示例。
                              "mask": MASK, "unk": UNK}}  # 执行当前语句以推进本节示例。
label_map32 = ["not_next", "is_next"]  # 计算并保存当前步骤的中间状态。
training_snapshot32 = {  # 计算并保存当前步骤的中间状态。
    "dataset": {"name": "toy-sentence-pairs-v1", "input_ids": pair_ids.tolist(),  # 执行当前语句以推进本节示例。
                "token_type_ids": pair_types.tolist(), "sentence_labels": pair_labels.tolist(),  # 执行当前语句以推进本节示例。
                "split": {"train": list(range(len(pair_ids))), "validation": [], "test": []}},  # 执行当前语句以推进本节示例。
    "vocab_sha256": stable_hash(vocab_spec),  # 执行当前语句以推进本节示例。
    "masking": {"strategy": "per-row-80-10-10", "mlm_probability": 0.35,  # 执行当前语句以推进本节示例。
                "seed": SEED + 9, "excluded_ids": sorted(SPECIAL_IDS)},  # 执行当前语句以推进本节示例。
    "recipe": {"optimizer": "AdamW", "steps": 80, "lr": 0.02, "weight_decay": 0.0,  # 执行当前语句以推进本节示例。
               "clip_grad_norm": 1.0, "sentence_weight": 0.7, "seed": SEED + 1,  # 执行当前语句以推进本节示例。
               "purpose": "controlled-overfit"},  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
package = package_bert(  # 计算并保存当前步骤的中间状态。
    bert, vocab_spec, label_map32, "nlp-lab/bert-demo",  # 执行当前语句以推进本节示例。
    "scratch-bert-demo", "1.0.0", training_snapshot32,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
PUBLISHER_REGISTRY32 = MappingProxyType({  # 计算并保存当前步骤的中间状态。
    ("scratch-bert-demo", "1.0.0"): package["manifest_sha256"]  # 执行当前语句以推进本节示例。
})  # 执行当前语句以推进本节示例。


def load_bert(package, expected_subject):  # 定义本节可复用的核心函数。
    if not isinstance(package, dict) or set(package) != {"manifest", "manifest_sha256", "state_bytes"}:  # 按当前条件选择后续控制路径。
        raise ValueError("BERT package 字段错误")  # 遇到非法合同立即显式失败。
    manifest = package["manifest"]  # 计算并保存当前步骤的中间状态。
    if not isinstance(manifest, dict):  # 按当前条件选择后续控制路径。
        raise ValueError("manifest 必须是字典")  # 遇到非法合同立即显式失败。
    key = (manifest.get("artifact_id"), manifest.get("version"))  # 计算并保存当前步骤的中间状态。
    expected_digest = PUBLISHER_REGISTRY32.get(key)  # 计算并保存当前步骤的中间状态。
    if expected_digest is None:  # 按当前条件选择后续控制路径。
        raise PermissionError("artifact id/version 未注册")  # 遇到非法合同立即显式失败。
    computed_digest = stable_hash(manifest)  # 计算并保存当前步骤的中间状态。
    if package["manifest_sha256"] != computed_digest:  # 按当前条件选择后续控制路径。
        raise ValueError("package 内 manifest hash 不一致")  # 遇到非法合同立即显式失败。
    if computed_digest != expected_digest:  # 按当前条件选择后续控制路径。
        raise PermissionError("package 内容不匹配发布者 registry")  # 遇到非法合同立即显式失败。
    if manifest.get("subject") != expected_subject:  # 按当前条件选择后续控制路径。
        raise PermissionError("业务主体不匹配")  # 遇到非法合同立即显式失败。
    if hashlib.sha256(package["state_bytes"]).hexdigest() != manifest["state_bytes_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("原始 state bytes 指纹不匹配")  # 遇到非法合同立即显式失败。
    validate_bert_manifest(manifest)  # 执行当前语句以推进本节示例。
    state = torch.load(io.BytesIO(package["state_bytes"]), map_location="cpu", weights_only=True)  # 计算并保存当前步骤的中间状态。
    if bert_tensor_state_hash(state) != manifest["state_tensor_sha256"]:  # 按当前条件选择后续控制路径。
        raise ValueError("tensor key/dtype/shape/bytes 指纹不匹配")  # 遇到非法合同立即显式失败。
    model = ScratchBert(**manifest["config"])  # 计算并保存当前步骤的中间状态。
    model.load_state_dict(state, strict=True)  # 计算并保存当前步骤的中间状态。
    return model.eval()  # 返回当前分支计算出的结果。


restored_bert = load_bert(package, "nlp-lab/bert-demo")  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    assert torch.equal(bert(pair_ids, pair_types, pair_mask)["mlm_logits"],  # 用受控断言验证关键不变量。
                       restored_bert(pair_ids, pair_types, pair_mask)["mlm_logits"])  # 执行当前语句以推进本节示例。

# 整体模型、vocab 或 config 被替换并重算内部 manifest hash，仍无法改变 registry。
forged_bert = ScratchBert(**bert.config)  # 计算并保存当前步骤的中间状态。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    for parameter in forged_bert.parameters():  # 遍历输入元素以累积或检查结果。
        parameter.zero_()  # 执行当前语句以推进本节示例。
fully_resigned = package_bert(  # 计算并保存当前步骤的中间状态。
    forged_bert, vocab_spec, label_map32, "nlp-lab/bert-demo",  # 执行当前语句以推进本节示例。
    "scratch-bert-demo", "1.0.0", training_snapshot32,  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
resigned_vocab = copy.deepcopy(package)  # 计算并保存当前步骤的中间状态。
resigned_vocab["manifest"]["vocab"]["tokens"][5:7] = list(reversed(  # 计算并保存当前步骤的中间状态。
    resigned_vocab["manifest"]["vocab"]["tokens"][5:7]  # 执行当前语句以推进本节示例。
))  # 执行当前语句以推进本节示例。
resigned_vocab["manifest"]["training_snapshot"]["vocab_sha256"] = stable_hash(  # 计算并保存当前步骤的中间状态。
    resigned_vocab["manifest"]["vocab"]  # 执行当前语句以推进本节示例。
)  # 执行当前语句以推进本节示例。
resigned_vocab["manifest_sha256"] = stable_hash(resigned_vocab["manifest"])  # 计算并保存当前步骤的中间状态。
resigned_config = copy.deepcopy(package)  # 计算并保存当前步骤的中间状态。
resigned_config["manifest"]["config"]["max_len"] += 1  # 计算并保存当前步骤的中间状态。
resigned_config["manifest"]["preprocess"]["max_len"] += 1  # 计算并保存当前步骤的中间状态。
resigned_config["manifest_sha256"] = stable_hash(resigned_config["manifest"])  # 计算并保存当前步骤的中间状态。
for candidate in (fully_resigned, resigned_vocab, resigned_config):  # 遍历输入元素以累积或检查结果。
    try:  # 尝试执行可能失败的受控操作。
        load_bert(candidate, "nlp-lab/bert-demo")  # 执行当前语句以推进本节示例。
        raise AssertionError("整体重签伪造必须被 registry 拒绝")  # 遇到非法合同立即显式失败。
    except PermissionError as exc:  # 捕获预期异常并验证失败分支。
        assert "registry" in str(exc)  # 用受控断言验证关键不变量。

bad_vocab = copy.deepcopy(vocab_spec)  # 计算并保存当前步骤的中间状态。
bad_vocab["tokens"] = bad_vocab["tokens"][:-1]  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    package_bert(  # 执行当前语句以推进本节示例。
        bert, bad_vocab, label_map32, "nlp-lab/bert-demo",  # 执行当前语句以推进本节示例。
        "bad-vocab", "1.0.0", training_snapshot32,  # 执行当前语句以推进本节示例。
    )  # 执行当前语句以推进本节示例。
    raise AssertionError("vocab/config mismatch 必须在发布前拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "vocab_size" in str(exc)  # 用受控断言验证关键不变量。

try:  # 尝试执行可能失败的受控操作。
    load_bert(package, "other-team")  # 执行当前语句以推进本节示例。
    raise AssertionError("跨主体加载不应通过")  # 遇到非法合同立即显式失败。
except PermissionError:  # 捕获预期异常并验证失败分支。
    pass  # 调整当前循环或占位控制流。
print({"artifact_registry": dict(PUBLISHER_REGISTRY32),  # 执行当前语句以推进本节示例。
       "resigned_model_vocab_config_rejected": True,  # 执行当前语句以推进本节示例。
       "vocab_tokens": len(vocab_spec["tokens"]),  # 执行当前语句以推进本节示例。
       "snapshot_rows": len(training_snapshot32["dataset"]["input_ids"])})  # 执行当前语句以推进本节示例。

## 9. 复杂度、常见失败与生产边界

- 每层 self-attention 时间约 $O(BT^2D)$，FFN 约 $O(BTD\,D_{ff})$；长文本可先切块或选择稀疏结构，但必须重新定义跨块信息流。
- 80/10/10 中“保持原词”的 10% 也要进入 label；只对实际 `[MASK]` 位置求 loss 是常见错误。
- special/pad 被 mask、segment 边界错一位、用 causal mask 训练 BERT、padding key 可见，都会改变任务定义。
- 动态 masking 要记录样本 id 与 RNG 策略以支持复现；分布式 worker 不应意外使用相同随机流。
- 线上特征必须复用训练 tokenizer、Unicode 归一化和最大长度策略；模型版本与词表版本不可独立漂移。
- 本实现没有大规模数据管道、混合精度、分布式 checkpoint 或下游微调，不能据此估算生产吞吐。

## 10. 原始论文与官方资料

- Devlin et al., [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)：MLM、句对任务、输入表示与 80/10/10 策略。
- Vaswani et al., [Attention Is All You Need](https://arxiv.org/abs/1706.03762)：多头注意力与 Encoder 基础。
- PyTorch 官方文档：[Embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)、[LayerNorm](https://pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html)、[cross_entropy](https://pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)。

链接用于追溯定义；本笔记只复现最小结构和合同，不声称达到原论文训练规模与基准结果。